# Preprocesamiento y features v1

Se construye un dataset supervisado desde `dataset_groundtruth.csv`. Se imputan valores numéricos con la mediana y categóricos con la moda del train. El texto faltante se representa como una cadena vacía. Las métricas de engagement reciben `log(1+x)` y se crean likes/vistas y comentarios/vistas. Los ratios quedan indefinidos cuando faltan conteos o las vistas son cero y luego se imputan con la mediana de train.

El primer fold de `StratifiedGroupKFold` (5 folds, semilla 42) evita que un canal aparezca en train y test. `video_id`, `channel_id` y `split` se exportan como metadatos para los experimentos posteriores y **no entran en el modelo**: `channel_id` se usa únicamente para agrupar la validación cruzada por canal dentro de train. También se excluyen predicción/etiquetado de origen y procedencia.

In [1]:
from pathlib import Path
import sys

from IPython.display import display

notebook_dir = Path.cwd()
if not (notebook_dir / "preprocessing.py").exists():
    notebook_dir = notebook_dir / "data_processing"
if not (notebook_dir / "preprocessing.py").exists():
    raise FileNotFoundError("No se encontró data_processing/preprocessing.py")
sys.path.insert(0, str(notebook_dir.resolve()))

from preprocessing import DEFAULT_INPUT_PATH, DEFAULT_OUTPUT_PATH, build_features

print(f"Entrada: {DEFAULT_INPUT_PATH}")
print(f"Salida:  {DEFAULT_OUTPUT_PATH}")

Entrada: C:\Users\LENOVO\MachineLearning\ML_Course\Proyecto\Machine-Learning-Proyect\dataset_disturbed_youtube _for_kids\dataset_groundtruth.csv
Salida:  C:\Users\LENOVO\MachineLearning\ML_Course\Proyecto\Machine-Learning-Proyect\data_processing\dataset_features_v1.csv


## 1. Inspección inicial

Se valida el esquema requerido, unicidad de `video_id` y valores faltantes antes de transformar.

In [2]:
import pandas as pd

raw = pd.read_csv(DEFAULT_INPUT_PATH, keep_default_na=False)
print(f"Filas: {len(raw):,} | Columnas: {len(raw.columns)}")
missing_counts = raw.apply(lambda column: column.astype("string").str.strip().eq("").sum())
display(missing_counts.loc[lambda values: values.gt(0)].rename("faltantes").to_frame())
display(raw["classification_label"].value_counts().rename("videos").to_frame())
assert raw["video_id"].is_unique

Filas: 4,797 | Columnas: 22


,faltantes
description,534
tags,882
duration_iso,1
duration_seconds,1
definition,1
caption,1
licensed_content,1
view_count,3
like_count,128
dislike_count,128


,videos
classification_label,
irrelevant,1936
suitable,1513
disturbing,929
restricted,419


## 2. Ingeniería de características, imputación y exportación

La función central conserva título, descripción y tags; codifica categorías como texto; mantiene duración en segundos; transforma vistas, likes, dislikes y comentarios; y calcula los dos ratios solicitados. Los campos `prediction`, `is_ground_truth`, `source_dataset`, `video_id`, `channel_id`, `channel_title`, `published_at`, `duration_iso` y `favorite_count` no se exportan como features (`video_id` y `channel_id` sí se exportan, pero solo como metadatos). `favorite_count` es constante (cero) en el dataset inspeccionado.

In [3]:
features, audit = build_features()
display(pd.Series(audit["missing_before"], name="faltantes antes de imputar").to_frame())
display(pd.Series(audit["numeric_imputation"], name="mediana de train").to_frame())
display(pd.Series(audit["categorical_imputation"], name="moda de train").to_frame())
display(pd.Series(audit["split_counts"], name="filas").to_frame())
print(f"Dataset guardado: {audit['output_path']}")
display(features.head())

,faltantes antes de imputar
description,534
tags,882
duration_iso,1
duration_seconds,1
definition,1
caption,1
licensed_content,1
view_count,3
like_count,128
dislike_count,128


,mediana de train
duration_seconds,349.000000
view_count,91776.000000
like_count,507.500000
dislike_count,54.000000
comment_count,49.000000
likes_per_view,0.005342
comments_per_view,0.000358


,moda de train
category_id,24
definition,hd
caption,false
licensed_content,false


,filas
train,3842
test,955


Dataset guardado: C:\Users\LENOVO\MachineLearning\ML_Course\Proyecto\Machine-Learning-Proyect\data_processing\dataset_features_v1.csv


,video_id,channel_id,split,title,description,tags,category_id,definition,caption,licensed_content,duration_seconds,view_count_log1p,like_count_log1p,dislike_count_log1p,comment_count_log1p,likes_per_view,comments_per_view,classification_label
0,RYq0wVVXSJo,UC_Q51pfEm5-L7Q7N-i1W3wg,train,✅ Frozen Princess Elsa The Mermaid Puzzle Game...,Jigsaw puzzle children game with Disney Frozen...,video games|puzzles|puzzle games for kids|puzz...,20,hd,false,true,150.0,11.705420,4.718499,3.891820,0.000000,0.000916,0.000000,suitable
1,w0xGZkViRYU,UCAqHpEnYCNEbBSOtSKI6D1A,train,Kids Master Cooking Play Toy Kitchen A Fun Kit...,Subscribe ....Like .....Share amazon : http:/...,toys|toy car|die-cast|cars playset|car smashes...,1,sd,false,false,46.0,14.359940,6.775366,5.638355,3.496508,0.000508,0.000019,suitable
2,NoaicfO02bA,UCYEHNnr_6XP-NY738hwXL1g,test,Маргарита и Лера играют в хокей,Маргарита и Лера играют в хокей\r\nМаргарита с...,маргарита|лера|девочки|дети|сестрички|подружки...,24,hd,false,true,607.0,10.987511,4.983607,3.806662,2.197225,0.002452,0.000135,suitable
3,5pd5Lv71Nm0,UC0AeIM8xtiFAgT_N1inUQxQ,test,Undress Display Model Spider-Man Costume Replica,Here's how the suit is undressed from the mann...,spiderman costume|spiderman suit|spider man co...,1,hd,false,true,91.0,13.379355,6.655440,5.327876,3.526361,0.001200,0.000051,suitable
4,PUHJKlyFD3c,UCxSjoRRrprrbX9uYIRVrerg,test,Frozen Elsa Confused Anna is Pregnant with Ryd...,,,22,hd,false,false,964.0,6.610696,2.484907,0.693147,0.000000,0.014825,0.000000,disturbing


## 3. Verificaciones del artefacto

El CSV `dataset_features_v1.csv` incluye la etiqueta objetivo y las columnas de metadatos `video_id`, `channel_id` y `split` (no son features); no incluye columnas de leakage ni faltantes. Se verifica además que ningún canal aparezca a la vez en train y test. Para una validación cruzada posterior, las imputaciones deben volver a ajustarse dentro de cada fold de entrenamiento y nunca reutilizar estadísticas calculadas fuera de ese fold.

In [4]:
exported = pd.read_csv(DEFAULT_OUTPUT_PATH, keep_default_na=False)
from preprocessing import METADATA_COLUMNS

for excluded in ("channel_title", "prediction", "is_ground_truth", "source_dataset", "published_at", "duration_iso", "favorite_count"):
    assert excluded not in exported.columns
assert {"classification_label", *METADATA_COLUMNS} <= set(exported.columns)
assert exported["video_id"].is_unique
train_channels = set(exported.loc[exported["split"] == "train", "channel_id"])
test_channels = set(exported.loc[exported["split"] == "test", "channel_id"])
assert not train_channels & test_channels, "Hay canales en train y test"
assert not exported.isna().any().any()
assert set(exported["split"]) == {"train", "test"}
n_features = len(exported.columns) - len(METADATA_COLUMNS) - 1
print(f"OK: {len(exported):,} filas; {n_features} features; sin nulos ni columnas de leakage; canales disjuntos entre train y test.")

OK: 4,797 filas; 14 features; sin nulos ni columnas de leakage; canales disjuntos entre train y test.
